# 🎙️ VoiceBatch v0.0 - [Turbo Speed Mode]
सीधे ड्राइव से मॉडल लोड होगा और हाई-स्पीड में ऑडियो जनरेट करेगा।

In [ ]:
# @title 🛠️ Step 1: इंस्टॉलेशन (Fast Setup)
import os
from google.colab import drive
print("⏳ जरूरी लाइब्रेरीज़ लोड हो रही हैं...")
!pip install -q coqpit-config coqui-tts gradio librosa soundfile
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
os.makedirs("outputs", exist_ok=True)
print("✅ सेटअप पूरा हुआ!")

In [ ]:
# @title 🚀 Step 2: हाई-स्पीड स्टूडियो लॉन्च करें
import gradio as gr
import torch, librosa, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/"

print("⏳ इंजन चालू हो रहा है...")
tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)

def turbo_gen(text, audio_sample):
    try:
        if not audio_sample: return None
        # स्क्रिप्ट को वाक्यों में तोड़ना (Speed Fix)
        parts = re.split(r'(?<=[।?!])\s+', text)
        final_wav = []
        
        for p in parts:
            if len(p.strip()) < 2: continue
            # तेज़ प्रोसेसिंग के लिए सीधे मेमोरी का उपयोग
            wav = tts.tts(text=p, speaker_wav=audio_sample, language='hi')
            final_wav.extend(wav)
        
        out_path = "outputs/VoiceBatch_Fast.wav"
        sf.write(out_path, np.array(final_wav), 24000)
        return out_path
    except Exception as e:
        return f"Error: {str(e)}"

gr.Interface(fn=turbo_gen, 
             inputs=[gr.Textbox(label="लिखी हुई कहानी", lines=10), 
                     gr.Audio(label="नमूना (Voice Sample)", type='filepath')],
             outputs=gr.Audio(label="तैयार आवाज़"),
             allow_flagging="never").launch(share=True, debug=True)